# Journey 2 — Build a SWAN workspace procedurally

**Learning goals:** Create a small SWAN grid, compose a validated configuration, and prepare a `ModelRun` in Python.

**Prerequisites:** [Journey 1](../journey_01_rompy_orientation/) and `rompy-swan` installed.


## Why this matters: SWAN data preparation

**Without Rompy:** a SWAN workflow typically requires separate scripts to resample bathymetry, align wind fields with the computational grid, extract an offshore spectral boundary, and write SWAN-specific text and binary files.

**With Rompy:** `SwanGrid`, `SwanDataGrid`, boundary components, and `ModelRun` keep those sources and transformations in one configuration. Rompy still needs the user's scientific choices—dataset, variables, coordinates, interpolation method, period, and quality checks—but it performs the repetitive staging and model-format conversion consistently.

The following cells verify the domain and make the source-to-model relationship visible.


!!! note
    This journey demonstrates configuration and workspace generation. The documentation build does not execute notebook cells or require a SWAN binary. Execution prerequisites are called out separately.


## 1. Define a small teaching domain

We use a short six-hour period and a compact regular domain near southwest Australia. The domain is deliberately small so the generated files are easy to inspect.


In [ ]:
from pathlib import Path
from rompy.core.time import TimeRange
from rompy_swan.grid import SwanGrid

time = TimeRange(
    start="2023-01-01T00:00:00",
    end="2023-01-01T06:00:00",
    interval="1h",
)
grid = SwanGrid(
    x0=115.0,
    y0=-32.0,
    rot=0.0,
    dx=0.25,
    dy=0.25,
    nx=9,
    ny=7,
)

print(time)
print("grid bounds:", grid.bbox())


## 2. Compose the model configuration

A SWAN configuration is a composition of model-specific components. The complete component setup is explored later; here the imports show the boundary between the core Rompy objects and the SWAN plugin.


In [ ]:
from rompy_swan.config import SwanConfig
from rompy_swan.components.cgrid import REGULAR
from rompy_swan.subcomponents.readgrid import GRIDREGULAR
from rompy_swan.subcomponents.spectrum import SPECTRUM

cgrid = REGULAR(
    grid=GRIDREGULAR(
        xp=grid.x0,
        yp=grid.y0,
        alp=grid.rot,
        xlen=grid.xlen,
        ylen=grid.ylen,
        mx=grid.nx - 1,
        my=grid.ny - 1,
    ),
    spectrum=SPECTRUM(mdc=36, flow=0.04, fhigh=1.0),
)

print(cgrid.render())

# A cgrid-only SwanConfig is valid and keeps this first lesson small.
config = SwanConfig(cgrid=cgrid)
print("validated config:", config.model_type)
print("rendered model fragment above; full workspace generation follows after data is connected")


## 3. Prepare a model run

The remaining configuration groups are added in the data and components lessons. Keeping `ModelRun` visible here makes the orchestration role explicit. The validated cgrid-only configuration already renders a model-native fragment.


In [ ]:
from rompy.model import ModelRun

# `config` is currently the validated minimal SwanConfig.
run = ModelRun(
    run_id="swan_first_run",
    period=time,
    output_dir=Path("swan_journey_workspace"),
    config=config,
)
# workspace = run.generate()


## Checkpoint

The grid and SWAN spectral grid are now defined, and the final `ModelRun` shape is clear. The generation call is left commented because later lessons add the data and component groups before creating a runnable workspace.

**Next:** [Configure SWAN declaratively with YAML](../journey_03_swan_declarative/).

**Reference:** [Existing procedural SWAN example](../example_procedural/).
